# Download cHL Dataset and Preprocessing

This tutorial will walk you to prepare examples data from the cHL dataset (Shaban et al., MAPS) to use KRONOS. 

### Marker Metadata

Before running KRONOS, we need to define marker metadata. Unlike RGB images, where channels are fixed (red, green, blue) and normalization values are often hardcoded, spatial proteomics (SP) datasets vary in the number and type of channels. Marker names also differ in formatting (e.g., “KI67” might appear as “Ki-67” or “KI-67”) To address this, KRONOS expects a CSV file containing metadata for all markers embedded in the inference data. An initial CSV with 175 markers is provided in [marker_metadata.csv](https://huggingface.co/MahmoodLab/KRONOS/blob/main/marker_metadata.csv).

The marker metadata CSV includes four columns:
- **marker_name**: The name of the marker in uppercase.
- **marker_id**: A unique identifier assigned to the marker in the pretraining dataset.
- **marker_mean**: The mean intensity value of the marker, calculated from a reference dataset (e.g., KRONOS pretraining dataset).
- **marker_std**: The standard deviation intensity value of the marker, also calculated from a reference dataset.

<br/>

### How Marker IDs are assigned to Markers
Marker IDs are assigned as integers from 1 to 512. In the pretrained dataset, nuclear markers are assigned IDs from 1 to 127, while non-nuclear markers receive IDs from 128 to 512. This grouping helps capture high-level similarities between markers of the same type. Within each category, markers are arranged alphabetically, but only even-numbered IDs are assigned to those included in the pretrained dataset. The odd-numbered IDs are intentionally left unassigned, reserved for biologically similar markers that were not part of the pretrained dataset. This approach allows end-users to assign marker IDs from the odd-numbered values, ensuring that any newly added markers remain closely linked to the existing structure while preserving biological relevance.

## Step 1: Download dataset and marker metadata

In [9]:
# Download cHL dataset (MAPS) and marker metadata 
from huggingface_hub import hf_hub_download
import shutil
import os
from utils.codex_dataset_prep import compute_stats

# Set project dir 
project_dir = "./codex_dataset"

# Download and prepare cHL dataset
stats = compute_stats('/nfs/turbo/umms-drjieliu/proj/HPAP-Spatial/CODEX/hpapdata', os.path.join(project_dir, "dataset"))

# Download marker_metadata.csv 
cached_file = hf_hub_download(
    repo_id="MahmoodLab/KRONOS",
    filename="marker_metadata.csv"
)
shutil.copy(cached_file, os.path.join(os.path.join(project_dir, "dataset"), "marker_metadata.csv"))

'./codex_dataset/dataset/marker_metadata.csv'

## Step 2: Matching pretraining markers with cHL data markers 

The following script maps the marker information (stored in `marker_info.csv`) from the original dataset to those used in the KRONOS pretraining dataset (`marker_metadata.csv`). <br />
It also displays a list of unmatched markers along with suggestions derived from marker name similarity with entries in `marker_metadata.csv`.

In [14]:
from utils import MarkerMetadata
# Define the project directory
project_dir = "./codex_dataset"  # Replace with your actual project directory
# Define paths for the dataset-specific marker info and the pretrained marker metadata files.
marker_info_csv_path = f"{project_dir}/dataset/marker_info.csv"        # Path to the dataset-specific marker info file.
marker_metadata_csv_path = f"{project_dir}/dataset/marker_metadata.csv"  # Path to the pretrained marker metadata file.
top_suggestions = 5  # Number of top suggestions to display for each unmatched marker.

# Create an instance of MarkerMetadata and retrieve the marker metadata.
obj = MarkerMetadata(marker_info_csv_path, marker_metadata_csv_path, top_suggestions)
obj.get_marker_metadata()

# Display the number of markers that do not match the pretrained dataset.
print(f"There are {len(obj.missing_marker_dict)} markers that do not match with the markers in the pretrained dataset.")

# Show the top suggestions based on marker name similarity for each unmatched marker.
print(f"Below are the top {top_suggestions} marker name similarity suggestions for each missing marker:")
display(obj.missing_marker_df)

# Display the dictionary for missing markers, which needs to be manually mapped to a biologically similar marker in marker_metadata.csv.
print("The following dictionary contains missing markers that need to be manually mapped:")
display(obj.missing_marker_dict)

There are 42 markers that do not match with the markers in the pretrained dataset.
Below are the top 5 marker name similarity suggestions for each missing marker:


,Suggestion 1,Suggestion 2,Suggestion 3,Suggestion 4,Suggestion 5
Missing Marker,,,,,
ACTA2,ACE2,GATA3,CTLA4,ATP5A,MCT
ARX,ATRX,PAX5,H2AX,SPARC,PROX1
ATP1A1,ATP5A,GATA3,NAKATP,ARID1A,PD1
CD117,CD11B,CD7,CD107A,CDT1,CD61
CD141,CD14,CD164,CD134,CD11B,CDT1
CD39L3,CD39,CD69,CD38,CD36,CD35
CD90,CD69,CD39,CD30,CD20,CD10
CDX2,CD2,CD28,CD27,CD25,CD23
CHGA,CGAS,CA9,C4A,IGA2,IGA1


The following dictionary contains missing markers that need to be manually mapped:


{'ACTA2': '',
 'ARX': '',
 'ATP1A1': '',
 'CD117': '',
 'CD141': '',
 'CD39L3': '',
 'CD90': '',
 'CDX2': '',
 'CHGA': '',
 'COL1A1': '',
 'COL4A1': '',
 'COL6': '',
 'CPEP': '',
 'DPP4': '',
 'ECAD': '',
 'GCG': '',
 'GHRL': '',
 'GP2': '',
 'HLA-DR': '',
 'HSPG2': '',
 'IRX2': '',
 'ISL1': '',
 'KRT': '',
 'LAM': '',
 'MCAM': '',
 'MMR': '',
 'NEUROD1': '',
 'NKX2-2': '',
 'NKX6-1': '',
 'NPY': '',
 'PAX6': '',
 'PD-L1': '',
 'PDX1': '',
 'PNLIP': '',
 'PPY': '',
 'PROINS': '',
 'SELP': '',
 'SOX9': '',
 'SST': '',
 'SYP': '',
 'TUBB3': '',
 'VIM': ''}

## Step 3: Manual marker mapping 

If some markers do not match based on their names, you can manually adjust the mapping. Use the provided suggestions and/or the list of marker names in the marker_metadata.csv file. <br/>
Simply copy the dictionary syntax from the previous step and update the values for the unmatched markers with a valid, biologically similar marker from the suggestions or the `marker_metadata.csv` file.

In [15]:
obj.missing_marker_dict |= {
    "ACTA2": "A-SMA",
    "ECAD": "E-CADHERIN",
    "KRT": "CYTOKERATIN",
    "LAM": "LAMINA",
    "HLA-DR": "HLA_DR",
    "PD-L1": "PDL1",
    "VIM": "VIMENTIN",
    "MMR": "CD206",
}

# Retrieve marker metadata using the updated mapping.
obj.get_marker_metadata_with_mapping()

if len(obj.missing_marker_dict) > 0:
    # Display the count of markers that still do not match the pretrained dataset.
    print(f"There are {len(obj.missing_marker_dict)} markers that still do not match the markers in the pretrained dataset.")

    # Display the dataframe of unmatched markers.
    display(obj.missing_marker_df)

    # Display the dictionary of unmatched markers that require manual mapping.
    display(obj.missing_marker_dict)
else:
    print("All markers have been successfully mapped to the pretrained dataset.")

There are 34 markers that still do not match the markers in the pretrained dataset.


,Suggestion 1,Suggestion 2,Suggestion 3,Suggestion 4,Suggestion 5
Missing Marker,,,,,
ARX,ATRX,PAX5,H2AX,SPARC,PROX1
ATP1A1,ATP5A,GATA3,NAKATP,ARID1A,PD1
CD117,CD11B,CD7,CD107A,CDT1,CD61
CD141,CD14,CD164,CD134,CD11B,CDT1
CD39L3,CD39,CD69,CD38,CD36,CD35
CD90,CD69,CD39,CD30,CD20,CD10
CDX2,CD2,CD28,CD27,CD25,CD23
CHGA,CGAS,CA9,C4A,IGA2,IGA1
COL1A1,HLA1,CD1A,COLLAGEN,CTLA4,CD11B


{'ARX': '',
 'ATP1A1': '',
 'CD117': '',
 'CD141': '',
 'CD39L3': '',
 'CD90': '',
 'CDX2': '',
 'CHGA': '',
 'COL1A1': '',
 'COL4A1': '',
 'COL6': '',
 'CPEP': '',
 'DPP4': '',
 'GCG': '',
 'GHRL': '',
 'GP2': '',
 'HSPG2': '',
 'IRX2': '',
 'ISL1': '',
 'MCAM': '',
 'NEUROD1': '',
 'NKX2-2': '',
 'NKX6-1': '',
 'NPY': '',
 'PAX6': '',
 'PDX1': '',
 'PNLIP': '',
 'PPY': '',
 'PROINS': '',
 'SELP': '',
 'SOX9': '',
 'SST': '',
 'SYP': '',
 'TUBB3': ''}

## Step 4 (Optional): Manually Set Metadata
If some markers are still unmatched with the pretrained dataset and you can not ignore these marker then you can manually assign their marker ID, mean, and standard deviation values:

- **Marker ID**: Choose an unassigned ID from the range 1–512 in marker_metadata.csv. Ideally, select an ID close to a biologically similar marker.
- **Mean & Std Values**: Calculate these from your dataset for the corresponding markers. Ensure marker intensities are converted to float type and intensities are in range of 0-1 before computing the mean and standard deviation.

In [39]:
new_marker_id_map = {
    # --- Nuclear Markers (转录因子) ---
    "ARX": 9,
    "CDX2": 15,
    "ISL1": 43,
    "IRX2": 45,
    "NEUROD1": 49,
    "NKX2-2": 51,
    "NKX6-1": 53,
    "PAX6": 55,
    "PDX1": 57,
    "SOX9": 59,
    
    # --- Non-nuclear Markers (胞质/膜/基质/激素) ---
    "ATP1A1": 141,
    "TUBB3": 145,
    "CD117": 177,
    "CD141": 191,
    "DPP4": 225,
    "MCAM": 237,
    "CD90": 239,
    "CD39L3": 247,
    "SELP": 277,
    "COL1A1": 311,
    "COL4A1": 313,
    "COL6": 315,
    "HSPG2": 317,
    "CHGA": 321,
    "GCG": 323,
    "GHRL": 325,
    "GP2": 327,
    "NPY": 329,
    "PNLIP": 331,
    "PPY": 333,
    "SST": 335,
    "SYP": 337,
    "PROINS": 339,
    "CPEP": 341
}
marker_id_map = {marker: marker_id for marker, marker_id in zip(obj.marker_info['marker_name'], obj.marker_info['marker_id']) if marker_id != 0} | new_marker_id_map
marker_metadata_dict = {marker: {"marker_id": marker_id, "marker_mean": stats.loc[stats['marker_name'] == marker, 'marker_mean'].item(), "marker_std": stats.loc[stats['marker_name'] == marker, 'marker_std'].item()} for marker, marker_id in marker_id_map.items()}

obj.set_marker_metadata(marker_metadata_dict)
# Display the count of markers that still do not match the pretrained dataset.
if len(obj.missing_marker_dict) > 0:
    print(f"There are {len(obj.missing_marker_dict)} markers that still do not match the markers in the pretrained dataset.")
    
    # Display the dataframe of unmatched markers.
    display(obj.missing_marker_df)

    # Display the dictionary of unmatched markers that require manual mapping.
    display(obj.missing_marker_dict)
else:
    print("All markers now have valid metadata.")
display(obj.marker_info)

All markers now have valid metadata.


,channel_id,marker_name,marker_mean,marker_std,marker_id
0,0,ACTA2,0.006497,0.033024,130
1,1,ARX,0.000851,0.002263,9
2,2,ATP1A1,0.001557,0.007289,141
3,3,CD117,0.000894,0.012016,177
4,4,CD11C,0.008051,0.035412,182
...,...,...,...,...,...
56,56,SST,0.003138,0.009973,335
57,57,SYP,0.007503,0.034891,337
58,58,TUBB3,0.000233,0.001186,145
59,59,VIM,0.006587,0.039536,496


## Step 5: Save Final Dataset Specific Metadata File

In [40]:
output_csv_path = f"{project_dir}/dataset/marker_info_with_metadata.csv"
obj.export_marker_metadata(output_csv_path)
display(obj.marker_info)

Exported marker metadata to ./codex_dataset/dataset/marker_info_with_metadata.csv


,channel_id,marker_name,marker_mean,marker_std,marker_id
0,0,ACTA2,0.006497,0.033024,130
1,1,ARX,0.000851,0.002263,9
2,2,ATP1A1,0.001557,0.007289,141
3,3,CD117,0.000894,0.012016,177
4,4,CD11C,0.008051,0.035412,182
...,...,...,...,...,...
56,56,SST,0.003138,0.009973,335
57,57,SYP,0.007503,0.034891,337
58,58,TUBB3,0.000233,0.001186,145
59,59,VIM,0.006587,0.039536,496
